# 09 - Quelles bandes séparent l'eau des nuages et de leurs ombres ?

Question du mentor : une ombre de nuage est sombre, comme l'eau. Avec seulement les couleurs
(RGB), comment les distinguer ? Et les autres bandes apportent-elles un signal qui fait la
différence ?

Expérience : **le même petit U-Net, entraîné de la même façon** (60 époques, graine 0, mêmes
découpes), en ne changeant **que les bandes d'entrée** :

| Configuration | Bandes | Idée physique |
|---|---|---|
| RGB | B4, B3, B2 | ce que verrait un appareil photo |
| RGB + proche infrarouge | + B8 | l'eau absorbe le proche infrarouge, la végétation le renvoie fortement |
| RGB + PIR + infrarouge moyen | + B11, B12 | l'eau est quasi noire en infrarouge moyen ; un sol à l'ombre garde du signal |
| 13 bandes optiques | B1 à B12, B8A | tout Sentinel-2 (modèle de secours) |
| radar seul | VV, VH | traverse les nuages, ignore les ombres, mais bruité (chatoiement) |
| optique + radar | 13 + 2 | modèle fusionné (notebook 08) |

Mesures, toutes sur des pays jamais vus à l'entraînement :
- **IoU de l'eau** en validation et en test (1 = carte parfaite) ;
- la même IoU sur les **images nuageuses** (au moins 20 % de la zone cachée), là où sont les ombres ;
- les **fausses détections d'eau** sur ces images nuageuses : part des pixels de terre (annotés
  « pas d'eau ») que le modèle prend pour de l'eau. Les ombres de nuages tombent justement sur
  ces images : c'est l'erreur visée par la question.

Exécution : Colab avec GPU T4, *Exécution > Tout exécuter* (environ 1 h 30 : 6 entraînements).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src import donnees, modeles, entrainement

noms = donnees.telecharger("data", radar=True)
APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
print(len(noms), "images | calcul sur", APPAREIL)
if APPAREIL == "cpu":
    print("ATTENTION : pas de GPU. Exécution > Modifier le type d'exécution > GPU T4, puis relancer.")

## 1. Données (13 bandes optiques + radar VV, VH)

In [ ]:
images, labels, perms = [], [], []
for nom in noms:
    s2, label, perm = donnees.lire_image(nom, "data")
    s1 = donnees.lire_radar(nom, "data")
    images.append(np.concatenate([s2, s1 * 100]).astype(np.int16))   # radar en centièmes de dB
    labels.append(label.astype(np.int8))
    perms.append(perm)

df = pd.DataFrame({"image": noms, "partie": [donnees.partie(n) for n in noms]})
df["nuages_annotes"] = [float((l < 0).mean()) for l in labels]
idx = {p: df.index[df["partie"] == p].tolist() for p in ["train", "validation", "test"]}
nuageuses = (df["nuages_annotes"] >= 0.2).to_numpy()
idx_nuageuses = {p: [k for k in idx[p] if nuageuses[k]] for p in ["validation", "test"]}
# statistiques calculées sur les 15 bandes (les pixels vus se repèrent sur l'optique), puis découpées
moyenne, ecart = entrainement.statistiques_bandes(images, idx["train"])
print({p: len(i) for p, i in idx.items()}, "| images nuageuses :", {p: len(i) for p, i in idx_nuageuses.items()})

## 2. Les six configurations

In [ ]:
B = {"B1": 0, "B2": 1, "B3": 2, "B4": 3, "B5": 4, "B6": 5, "B7": 6, "B8": 7, "B8A": 8, "B9": 9,
     "B10": 10, "B11": 11, "B12": 12, "VV": 13, "VH": 14}
CONFIGS = {
    "RGB": ["B4", "B3", "B2"],
    "RGB + proche infrarouge": ["B4", "B3", "B2", "B8"],
    "RGB + PIR + infrarouge moyen": ["B4", "B3", "B2", "B8", "B11", "B12"],
    "13 bandes optiques": list(B)[:13],
    "radar seul": ["VV", "VH"],
    "optique + radar": list(B),
}
EPOQUES = 60 if APPAREIL == "cuda" else 2
GRAINE = 0

def fausses_detections(probs, indices, seuil=0.5):
    """Part des pixels annotés « terre » que le modèle prend pour de l'eau."""
    fp = terre = 0
    for k in indices:
        t = labels[k] == 0
        fp += int(((probs[k] > seuil) & t).sum()); terre += int(t.sum())
    return fp / terre if terre else 0.0

os.makedirs("modeles/bandes", exist_ok=True); os.makedirs("resultats", exist_ok=True)
lignes, predictions = [], {}
for nom, bandes in CONFIGS.items():
    b = [B[x] for x in bandes]
    sous = [im[b] for im in images]
    entrainement.fixer_graine(GRAINE)
    m = modeles.Normalise(modeles.PetitUNet(canaux=len(b)), moyenne[b], ecart[b])
    chemin = f"modeles/bandes/{nom.replace(' ', '_').replace('+', 'plus')}_{EPOQUES}ep"
    if os.path.exists(chemin + ".pt"):
        m.load_state_dict(torch.load(chemin + ".pt", map_location="cpu")); m.to(APPAREIL)
    else:
        print(f"--- {nom} ({len(b)} bandes) ---")
        h = entrainement.entrainer(m, sous, labels, idx["train"], idx["validation"], APPAREIL,
                                   epoques=EPOQUES, graine=GRAINE, journal=lambda t: None)
        torch.save(m.state_dict(), chemin + ".pt"); h.to_csv(chemin + ".csv", index=False)
    P = entrainement.predire(m, sous, idx["validation"] + idx["test"], APPAREIL)
    predictions[nom] = P
    lignes.append({
        "configuration": nom, "bandes": len(b),
        "iou_validation": round(entrainement.iou_eau(P, labels, idx["validation"]), 3),
        "iou_test": round(entrainement.iou_eau(P, labels, idx["test"]), 3),
        "iou_nuageuses_validation": round(entrainement.iou_eau(P, labels, idx_nuageuses["validation"]), 3),
        "iou_nuageuses_test": round(entrainement.iou_eau(P, labels, idx_nuageuses["test"]), 3),
        "fausses_detections_nuageuses_validation": round(fausses_detections(P, idx_nuageuses["validation"]), 4),
        "fausses_detections_nuageuses_test": round(fausses_detections(P, idx_nuageuses["test"]), 4),
    })
    print(lignes[-1])
    del sous

bandes_res = pd.DataFrame(lignes).set_index("configuration")
bandes_res.to_csv("resultats/bandes_09.csv")
bandes_res

## 3. Lecture des résultats

Graphique de gauche : qualité de la carte d'eau (plus haut = mieux). Graphique de droite :
fausses détections sur les images nuageuses (plus bas = mieux). Une seule graine par
configuration : les écarts de moins de 0,01 à 0,02 d'IoU ne sont pas significatifs (voir la
variabilité entre graines des notebooks 05 et 08).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
y = np.arange(len(bandes_res))
ax[0].barh(y - 0.2, bandes_res["iou_test"], 0.4, label="test, toutes les images", color="#2b6cb0")
ax[0].barh(y + 0.2, bandes_res["iou_nuageuses_test"], 0.4, label="test, images nuageuses", color="#90cdf4")
ax[0].set_yticks(y, bandes_res.index); ax[0].invert_yaxis(); ax[0].set_xlabel("IoU de l'eau (1 = parfait)")
ax[0].legend(loc="lower right"); ax[0].set_xlim(0, 1)
ax[1].barh(y, 100 * bandes_res["fausses_detections_nuageuses_test"], color="#c53030")
ax[1].set_yticks(y, bandes_res.index); ax[1].invert_yaxis()
ax[1].set_xlabel("% des pixels de terre pris pour de l'eau (images nuageuses, test)")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("resultats/bandes_09.png", dpi=150); plt.show()

## 4. Sur une image nuageuse du test

L'image du test la plus nuageuse parmi celles qui contiennent de l'eau : ce que voit chaque
configuration. Rouge : eau détectée à tort (fausse détection) ; bleu : eau bien détectée ; jaune :
eau manquée ; gris : zone non annotée (nuages).

In [ ]:
cand = [k for k in idx_nuageuses["test"] if (labels[k] == 1).mean() > 0.02]
k = max(cand or idx_nuageuses["test"] or idx["test"], key=lambda i: df.loc[i, "nuages_annotes"])
choix = ["RGB", "RGB + PIR + infrarouge moyen", "13 bandes optiques", "optique + radar"]
fig, ax = plt.subplots(1, len(choix) + 1, figsize=(4.2 * (len(choix) + 1), 4.6))
ax[0].imshow(donnees.rgb(images[k][:13].astype(np.float32))); ax[0].set_title(df.loc[k, "image"])
for a, nom in zip(ax[1:], choix):
    p = predictions[nom][k].astype(np.float32) > 0.5
    l = labels[k]
    carte = np.ones(l.shape + (3,)) * 0.92
    carte[l < 0] = 0.6
    carte[(l == 1) & p] = (0.17, 0.42, 0.69)
    carte[(l == 1) & ~p] = (0.95, 0.77, 0.1)
    carte[(l == 0) & p] = (0.77, 0.19, 0.19)
    a.imshow(carte); a.set_title(nom, fontsize=10)
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.savefig("resultats/bandes_09_exemple.png", dpi=150); plt.show()

## 5. Récupérer les résultats

Un seul fichier compressé (tableau, figures, historiques d'entraînement), téléchargé
automatiquement depuis Colab.

In [ ]:
import shutil, glob
os.makedirs("export_09", exist_ok=True)
for f in glob.glob("resultats/bandes_09*") + glob.glob("modeles/bandes/*.csv"):
    shutil.copy(f, "export_09/")
shutil.make_archive("resultats_bandes_09", "zip", "export_09")
try:
    from google.colab import files
    files.download("resultats_bandes_09.zip")
except ImportError:
    print("Hors de Colab : resultats_bandes_09.zip")